# Notebook 04: Modern Causal ML (DR-Learner)

Classical uplift models are useful, but modern pipelines often use doubly robust methods for improved stability.

## Conceptual framing
DR-Learner combines:
- outcome nuisance models (`mu1`, `mu0`)
- treatment propensity model (`e(x)`)
to build pseudo-outcomes that can be more robust to nuisance model misspecification.

In [ ]:
from pathlib import Path
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='talk')

if Path.cwd().name == 'notebooks':
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import set_seed, ensure_output_dirs, save_plot
from src.preprocessing import load_hillstrom_data, basic_cleaning, split_features_outcomes_treatment, create_train_valid_test_split
from src.meta_learners import XLearner
from src.modern_causal import DRLearnerWrapper, OrthogonalForestWrapper
from src.evaluation import qini_curve, qini_coefficient, auuc_score, uplift_by_decile

SEED = 42
set_seed(SEED)
OUT_DIRS = ensure_output_dirs(PROJECT_ROOT)
DATA_PATH = PROJECT_ROOT / 'data' / 'hillstrom.csv'

In [ ]:
df = basic_cleaning(load_hillstrom_data(str(DATA_PATH)))
X, y, t = split_features_outcomes_treatment(df, outcome_col='conversion', treatment_col='treatment')
splits = create_train_valid_test_split(X, y, t, test_size=0.2, valid_size=0.2, random_state=SEED, stratify=True)
print('Split sizes:', {k: len(v) for k, v in splits.items() if k.startswith('X_')})

## 1) Propensity model and overlap diagnostics

In [ ]:
prop_model = LogisticRegression(max_iter=1000)
prop_model.fit(splits['X_train'], splits['t_train'])
prop_test = prop_model.predict_proba(splits['X_test'])[:, 1]

fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(prop_test[splits['t_test'] == 1], bins=40, alpha=0.6, label='treated', ax=ax)
sns.histplot(prop_test[splits['t_test'] == 0], bins=40, alpha=0.6, label='control', ax=ax)
ax.set_title('Propensity Score Overlap on Test Set')
ax.set_xlabel('Estimated propensity P(T=1|X)')
ax.set_ylabel('count')
ax.legend()
plt.tight_layout()
save_plot(fig, OUT_DIRS['figures'] / 'nb04_drlearner_overlap.png')
plt.show()

## 2) Fit DR-Learner and compare to best classical baseline (X-Learner)

In [ ]:
x_learner = XLearner(random_state=SEED).fit(splits['X_train'], splits['t_train'], splits['y_train'])
x_scores = x_learner.predict_uplift(splits['X_test'])

dr = DRLearnerWrapper(random_state=SEED, prefer_econml=True).fit(splits['X_train'], splits['t_train'], splits['y_train'])
dr_scores = dr.predict_uplift(splits['X_test'])
print('DR backend used:', dr.backend_)

In [ ]:
score_dict = {'x_learner': x_scores, 'dr_learner': dr_scores}
rows = []
for name, score in score_dict.items():
    rows.append({
        'model': name,
        'qini': qini_coefficient(splits['y_test'], splits['t_test'], score),
        'auuc': auuc_score(splits['y_test'], splits['t_test'], score),
    })
metrics = pd.DataFrame(rows).sort_values('qini', ascending=False).reset_index(drop=True)
display(metrics)
metrics.to_csv(OUT_DIRS['tables'] / 'nb04_drlearner_metrics.csv', index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
for name, score in score_dict.items():
    c = qini_curve(splits['y_test'], splits['t_test'], score)
    ax.plot(c['fraction_targeted'], c['incremental_outcomes'], label=name)
ax.set_title('Qini Curves: DR-Learner vs X-Learner')
ax.set_xlabel('Fraction targeted')
ax.set_ylabel('Cumulative incremental conversions')
ax.legend()
plt.tight_layout()
save_plot(fig, OUT_DIRS['figures'] / 'nb04_drlearner_vs_xlearner_qini.png')
plt.show()

## 3) Optional orthogonal forest validation (if econml available)

In [ ]:
of_scores = None
try:
    of = OrthogonalForestWrapper(random_state=SEED).fit(splits['X_train'], splits['t_train'], splits['y_train'])
    of_scores = of.predict_uplift(splits['X_test'])
    score_dict['orthogonal_forest'] = of_scores
    print('Orthogonal forest fitted successfully.')
except Exception as exc:
    print('Orthogonal forest skipped:', exc)

## 4) Robustness diagnostics

In [ ]:
rng = np.random.default_rng(SEED)
boot = []
y_test = np.asarray(splits['y_test'])
t_test = np.asarray(splits['t_test'])
dr_test = np.asarray(dr_scores)
for _ in range(200):
    idx = rng.integers(0, len(y_test), size=len(y_test))
    boot.append(qini_coefficient(y_test[idx], t_test[idx], dr_test[idx]))
ci_low, ci_high = np.quantile(boot, [0.025, 0.975])
print(f'DR-Learner Qini bootstrap 95% CI: [{ci_low:.4f}, {ci_high:.4f}]')

In [ ]:
test_eval = splits['X_test'].copy()
test_eval['pred_uplift_dr'] = dr_scores
if 'history' in test_eval.columns:
    test_eval['history_bucket'] = pd.qcut(test_eval['history'], q=4, duplicates='drop')
    subgroup = test_eval.groupby('history_bucket')['pred_uplift_dr'].agg(['mean', 'median', 'count']).reset_index()
    display(subgroup)
    subgroup.to_csv(OUT_DIRS['tables'] / 'nb04_drlearner_subgroup_history.csv', index=False)
else:
    print('history feature not available for subgroup summary.')

In [ ]:
deciles = []
for name, score in score_dict.items():
    d = uplift_by_decile(splits['y_test'], splits['t_test'], score)
    d['model'] = name
    deciles.append(d)
decile_df = pd.concat(deciles, ignore_index=True)
decile_df.to_csv(OUT_DIRS['tables'] / 'nb04_uplift_by_decile.csv', index=False)
display(decile_df.head(20))

## End-of-notebook summary
- DR-Learner introduces doubly robust structure through nuisance outcome + propensity modeling.
- Metric comparison to classical meta-learner baseline is reported above.
- Notebook 05 converts model ranking quality into budget-constrained policy and business impact simulation.